# Phase 6: Modelling and Hyperparameter Optimization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from scipy.stats import loguniform, uniform

# Configure Plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Load the Fully Engineered ML Dataset

In [2]:
processed_dir = Path("../data/processed")
features_path = processed_dir / "final_ml_features.parquet"

df = pd.read_parquet(features_path)

# Drop any NaN rows generated from the lagging/rolling logic (the first 168 hours will be NaN due to the t-168 weekly lag)
df = df.dropna()

print(f"Dataset shape after dropping Nulls: {df.shape}")
df.head(3)

Dataset shape after dropping Nulls: (10009, 35)


,total_demand,lag_1,lag_2,lag_3,lag_24,lag_48,lag_168,rolling_mean_6,rolling_std_6,rolling_max_6,...,ewma_span_72,hour,day_of_week,is_weekend,day_of_month,month,quarter,hour_sin,hour_cos,is_holiday
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-01-08 00:00:00,1079,2219.0,3740.0,5834.0,1042.0,1051.0,6610.0,5106.500000,1808.302381,7272.0,...,3867.045281,0,2,0,8,1,1,0.000000,1.000000,0
2025-01-08 01:00:00,382,1079.0,2219.0,3740.0,423.0,437.0,7481.0,4074.333333,2073.110963,5834.0,...,3790.660479,1,2,0,8,1,1,0.258819,0.965926,0
2025-01-08 02:00:00,200,382.0,1079.0,2219.0,242.0,231.0,6126.0,3173.666667,2337.745296,5834.0,...,3697.272521,2,2,0,8,1,1,0.500000,0.866025,0


## 2. Train / Test Split

In [3]:
# Sort the dataset chronologically just in case
df.sort_index(inplace=True)

# Determine the index cutoff
train_size = int(len(df) * 0.9)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

# The target variable is 'total_demand'. However, 'holiday_name' is categorical text. We will drop text columns.
DROP_COLS = ['total_demand']
FEATURE_COLS = [col for col in train_df.columns if col not in DROP_COLS]

X_train = train_df[FEATURE_COLS]
y_train = train_df['total_demand']

X_test = test_df[FEATURE_COLS]
y_test = test_df['total_demand']

X_train.to_parquet("../data/processed/X_train.parquet")
X_test.to_parquet("../data/processed/X_test.parquet")
y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")

print(f"Training Set: X={X_train.shape}, y={y_train.shape}")
print(f"Holdout Test Set: X={X_test.shape}, y={y_test.shape}")

Training Set: X=(9008, 34), y=(9008,)
Holdout Test Set: X=(1001, 34), y=(1001,)


## 3. Modeling Pipeline Construction

In [ ]:
tscv = TimeSeriesSplit(n_splits=10)

scorer = make_scorer(mean_absolute_error, greater_is_better=False)

def build_pipeline(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures()),
        ('reg', model)
    ])


def select_best_params(cv_results_, n_splits=10):
    """
    Apply 7 model selection strategies to RandomizedSearchCV cv_results_.

    Strategies (scores are negative MAE, so we negate to get positive MAE):
      0  Classic              – argmin E[MAE]
      1  Penalized Mean       – argmin(E[MAE] + λ·σ),  λ=1
      2  One Standard Error   – most stable model within best_MAE + SE
      3  Coeff of Variation   – argmin(E[MAE] · (1 + σ/E[MAE]))
      4  Upper Conf Bound     – argmin(E[MAE] + 1.96·σ/√n)
      5  CVaR (worst 20%)     – argmin(avg of worst 2 folds)
      6  Z-Score Combined     – argmin(z(mean) + z(std))
    """
    mean_mae = -cv_results_['mean_test_score']
    std_mae  =  cv_results_['std_test_score']
    n = n_splits

    fold_keys  = sorted(k for k in cv_results_ if k.startswith('split') and k.endswith('_test_score'))
    fold_maes  = -np.array([cv_results_[k] for k in fold_keys])  # (n_folds, n_candidates)

    strategies = {}

    # 0. Classic
    strategies['0_Classic'] = np.argmin(mean_mae)

    # 1. Penalized Mean (λ=1)
    strategies['1_Penalized_Mean'] = np.argmin(mean_mae + std_mae)

    # 2. One Standard Error Rule
    best_idx   = np.argmin(mean_mae)
    se         = std_mae[best_idx] / np.sqrt(n)
    within_1se = mean_mae <= (mean_mae[best_idx] + se)
    eligible   = np.where(within_1se, std_mae, np.inf)
    strategies['2_One_SE_Rule'] = np.argmin(eligible)

    # 3. Coefficient of Variation
    strategies['3_Coeff_Variation'] = np.argmin(mean_mae * (1 + std_mae / mean_mae))

    # 4. Upper Confidence Bound
    strategies['4_UCB'] = np.argmin(mean_mae + 1.96 * std_mae / np.sqrt(n))

    # 5. CVaR – average of worst 20% of folds (2 out of 10)
    k_worst        = max(1, int(np.ceil(0.2 * n)))
    worst_fold_mae = np.sort(fold_maes, axis=0)[-k_worst:, :]
    strategies['5_CVaR_worst20pct'] = np.argmin(worst_fold_mae.mean(axis=0))

    # 6. Z-Score Combined
    z_mean = (mean_mae - mean_mae.mean()) / (mean_mae.std() + 1e-8)
    z_std  = (std_mae  - std_mae.mean())  / (std_mae.std()  + 1e-8)
    strategies['6_ZScore_Combined'] = np.argmin(z_mean + z_std)

    return strategies


def display_selection_results(search, model_name, n_splits=10):
    """Print best params for each of the 7 selection strategies."""
    selections = select_best_params(search.cv_results_, n_splits=n_splits)
    print(f"\n{'='*65}")
    print(f"  {model_name} — Best Params by Selection Strategy")
    print(f"{'='*65}")
    for strategy, idx in selections.items():
        params = search.cv_results_['params'][idx]
        m = -search.cv_results_['mean_test_score'][idx]
        s =  search.cv_results_['std_test_score'][idx]
        print(f"\n  [{strategy}]")
        print(f"    Params : {params}")
        print(f"    MAE    : {m:.4f} ± {s:.4f}")


## 4. Hyperparameter Optimization: Regularized Models

In [ ]:
# 1. Ridge Regression Optimization
ridge_pipe = build_pipeline(Ridge(random_state=42))

ridge_param_dist = {
    'poly__degree': [1, 2, 3],
    'reg__alpha': loguniform(1e-2, 1e2),
}

print("Starting Ridge RandomizedSearch...")
ridge_search = RandomizedSearchCV(
    estimator=ridge_pipe,
    param_distributions=ridge_param_dist,
    n_iter=30,
    cv=tscv,
    scoring=scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

ridge_search.fit(X_train, y_train)
display_selection_results(ridge_search, "Ridge Regression")


In [ ]:
# 2. Lasso Regression Optimization
lasso_pipe = build_pipeline(Lasso(random_state=42, max_iter=2000))

lasso_param_dist = {
    'poly__degree': [1, 2, 3],
    'reg__alpha': loguniform(1e-3, 1e2),
}

print("Starting Lasso RandomizedSearch...")
lasso_search = RandomizedSearchCV(
    estimator=lasso_pipe,
    param_distributions=lasso_param_dist,
    n_iter=30,
    cv=tscv,
    scoring=scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

lasso_search.fit(X_train, y_train)
display_selection_results(lasso_search, "Lasso Regression")


In [ ]:
# 3. ElasticNet Regression Optimization
elasticnet_pipe = build_pipeline(ElasticNet(random_state=42, max_iter=2000))

elasticnet_param_dist = {
    'poly__degree': [1, 2, 3],
    'reg__alpha': loguniform(1e-3, 1e1),
    'reg__l1_ratio': uniform(0.1, 0.8),  # samples in [0.1, 0.9]
}

print("Starting ElasticNet RandomizedSearch...")
elasticnet_search = RandomizedSearchCV(
    estimator=elasticnet_pipe,
    param_distributions=elasticnet_param_dist,
    n_iter=40,
    cv=tscv,
    scoring=scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

elasticnet_search.fit(X_train, y_train)
display_selection_results(elasticnet_search, "ElasticNet Regression")


## 5. Hyperparameter Optimization: SGDRegressor

In [ ]:
# 4. SGDRegressor Optimization
sgd_pipe = build_pipeline(SGDRegressor(random_state=42))

sgd_param_dist = {
    'poly__degree': [1, 2],
    'reg__penalty': ['l2', 'l1', 'elasticnet'],
    'reg__alpha': loguniform(1e-4, 1e-1),
    'reg__learning_rate': ['invscaling', 'adaptive'],
}

print("Starting SGDRegressor RandomizedSearch...")
sgd_search = RandomizedSearchCV(
    estimator=sgd_pipe,
    param_distributions=sgd_param_dist,
    n_iter=40,
    cv=tscv,
    scoring=scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

sgd_search.fit(X_train, y_train)
display_selection_results(sgd_search, "SGDRegressor")
